In [1]:
import os
import torch
from tqdm import tqdm
# change working directory 
os.chdir( os.path.join( os.environ["VSC_DATA"], "physioex" ) )

from physioex.train.utils.fast_train import FastEvalDataset
from physioex.train.models import load_model
from physioex.train.networks.prototype import voting_strategy as proto_voting 
from physioex.train.networks.base import voting_strategy as voting 

from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score
import numpy as np

dataset = "sleepedf"
device = "cuda" if torch.cuda.is_available() else "cpu"

model_kwargs = {
    "in_channels": 3,
    "sequence_length": 21,
    "N" : 1,
    "S" : 2,
    "n_prototypes" : 50,
}


In [2]:
def load_dataset( dataset ):
    eval_dataset = FastEvalDataset(
        datasets = [dataset],
        preprocess = "xsleepnet",
        indexed_channels = [0, 1, 2],
        split = "test",
        data_folder=f"/readonly/{os.environ['VSC_SCRATCH_PROJECTS_BASE']}/2024_111/guido/"
    )

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=1,
        num_workers=1,
    )

    return eval_loader


def load_finetuned_models( dataset ):
    proto = load_model(
        model = "physioex.train.networks.prototypev1:ProtoSleepNetV1",
        model_kwargs = model_kwargs,
        ckpt_path = f"models/finetune/protosleepnetv1/{dataset}/EEG-EOG-EMG/model.ckpt",
        softmax = True,
        summary = False,
    ).eval()[0]

    transf = load_model(
        model = "physioex.train.networks.sleeptransformer:SleepTransformer",
        model_kwargs = model_kwargs,
        ckpt_path = f"models/finetune/sleeptransformer/{dataset}/EEG-EOG-EMG/model.ckpt",
        softmax = True,
        summary = False,
    ).eval()[0]
    
    return proto, transf

In [3]:
def occlusion_maskv1( inputs : torch.Tensor ):
    occlusion_mask = torch.zeros( inputs.shape[0], inputs.shape[1], inputs.shape[2] )
    # randomly set channels to 1
    # batch_size is set to 1 so we can avoid it
    for i in range(inputs.shape[1]):
        num_channels_to_occlude = torch.randint(1, 3, (1,)).item()
        channels_to_occlude = torch.randperm(inputs.shape[2])[:num_channels_to_occlude]
        occlusion_mask[:, i, channels_to_occlude] = 1
    return occlusion_mask

def occlusion_maskv2( inputs : torch.Tensor ):
    occlusion_mask = torch.zeros( inputs.shape[0], inputs.shape[1], inputs.shape[2] )
    # randomly set channels to 1
    # batch_size is set to 1 so we can avoid it
    for i in range(inputs.shape[0]):
        num_channels_to_occlude = torch.randint(1, 3, (1,)).item()
        channels_to_occlude = torch.randperm(inputs.shape[2])[:num_channels_to_occlude]
        occlusion_mask[i, :, channels_to_occlude] = 1
    return occlusion_mask



def forward( proto_model, transf_model, inputs, occlusion_mask, targets ):

    with torch.no_grad():
        inputs = inputs.to( device )
        occlusion_mask = occlusion_mask.to( device )
        inputs = torch.einsum("bsctf, bsc -> bsctf", inputs, occlusion_mask)
        _ , proto_outputs = proto_voting(proto_model, inputs, model_kwargs["sequence_length"] )
        _, transf_outputs = voting(transf_model, inputs, model_kwargs["sequence_length"] )

    proto_outputs = proto_outputs.detach().cpu().squeeze(0)
    transf_outputs = transf_outputs.detach().cpu().squeeze(0)
    targets = targets.cpu().squeeze(0)
    
    return proto_outputs, transf_outputs, targets

In [4]:
# use accuracy, precision, recall, f1-score and cohen-kappa as metrics


def eval_robustness( proto_model, transf_model, test_loader, occlusion_mask_fn ):
    """
    Evaluate the robustness of the model by occluding channels and measuring performance.
    """
    metrics_proto = {
        "accuracy" : .0,
        "precision" : .0,
        "recall" : .0,
        "f1_score" : .0,
        "cohen_kappa" : .0,
        "count" : 0,
    }

    metrics_transf = {
        "accuracy" : .0,
        "precision" : .0,
        "recall" : .0,
        "f1_score" : .0,
        "cohen_kappa" : .0,
        "count" : 0,
    }
    
    for batch in tqdm(test_loader):
        inputs, targets, subjects, dataset_idx = batch
        
        occlusion_mask = occlusion_mask_fn(inputs)

        proto_outputs, transf_outputs, targets = forward(
            proto_model, transf_model, inputs, occlusion_mask, targets
        )

        y_pred_proto = proto_outputs.numpy()
        y_pred_transf = transf_outputs.numpy()
        y_true = targets.numpy()

        # update metrics
        metrics_proto["accuracy"] += accuracy_score(y_true, np.argmax(y_pred_proto, axis=1))
        metrics_proto["precision"] += precision_score(y_true, np.argmax(y_pred_proto, axis=1), average='weighted', zero_division=0)
        metrics_proto["recall"] += recall_score(y_true, np.argmax(y_pred_proto, axis=1), average='weighted', zero_division=0)
        metrics_proto["f1_score"] += f1_score(y_true, np.argmax(y_pred_proto, axis=1), average='weighted', zero_division=0)
        metrics_proto["cohen_kappa"] += cohen_kappa_score(y_true, np.argmax(y_pred_proto, axis=1))
        metrics_proto["count"] += 1

        metrics_transf["accuracy"] += accuracy_score(y_true, np.argmax(y_pred_transf, axis=1))
        metrics_transf["precision"] += precision_score(y_true, np.argmax(y_pred_transf, axis=1), average='weighted', zero_division=0)
        metrics_transf["recall"] += recall_score(y_true, np.argmax(y_pred_transf, axis=1), average='weighted', zero_division=0)
        metrics_transf["f1_score"] += f1_score(y_true, np.argmax(y_pred_transf, axis=1), average='weighted', zero_division=0)
        metrics_transf["cohen_kappa"] += cohen_kappa_score(y_true, np.argmax(y_pred_transf, axis=1))
        metrics_transf["count"] += 1
    
    # compute the average metrics
    for key in metrics_proto:
        metrics_proto[key] /= metrics_proto["count"]
        metrics_transf[key] /= metrics_transf["count"]
    
    del metrics_proto["count"]
    del metrics_transf["count"]

    return metrics_proto, metrics_transf

In [8]:
import pandas as pd

results_df = []

n_trials = 10


for dataset in ["sleepedf", "hmc", "dcsm", "mass"]:
    print(f"Evaluating robustness for dataset: {dataset}")
    
    # Load the dataset
    test_loader = load_dataset(dataset)

    # Load the models
    proto_model, transf_model = load_finetuned_models(dataset)

    for i in range(n_trials):    
        # Evaluate robustness with occlusion mask v1
        print("Evaluating with occlusion mask v1...")
        metrics_proto_v1, metrics_transf_v1 = eval_robustness(
            proto_model, transf_model, test_loader, occlusion_maskv1
        )
        
        # create a directory for the results 
        metrics_proto_v1["dataset"] = dataset
        metrics_transf_v1["dataset"] = dataset
        
        metrics_proto_v1["occlusion_mask"] = "v1"
        metrics_transf_v1["occlusion_mask"] = "v1"
        
        metrics_proto_v1["model"] = "ProtoSleepNetV1"
        metrics_transf_v1["model"] = "SleepTransformer"    
        metrics_proto_v1["trial"] = i

        # merge the results in a dictionary
        results_df.append(metrics_proto_v1)
        results_df.append(metrics_transf_v1)
        
        print("Evaluating with occlusion mask v2...")
        metrics_proto_v2, metrics_transf_v2 = eval_robustness(
            proto_model, transf_model, test_loader, occlusion_maskv2
        )
        
        # create a directory for the results 
        metrics_proto_v2["dataset"] = dataset
        metrics_transf_v2["dataset"] = dataset

        metrics_proto_v2["occlusion_mask"] = "v2"
        metrics_transf_v2["occlusion_mask"] = "v2"

        metrics_proto_v2["model"] = "ProtoSleepNetV1"
        metrics_transf_v2["model"] = "SleepTransformer"
        metrics_proto_v2["trial"] = i

        # merge the results in a dictionary
        results_df.append(metrics_proto_v2)
        results_df.append(metrics_transf_v2)


results_df = pd.DataFrame(results_df)
results_df.head()

2025-06-09 14:01:51.337 | INFO     | physioex.train.utils.fast_train:__init__:70 - Loading FastEvalDataset for sleepedf with preprocess xsleepnet


Evaluating robustness for dataset: sleepedf
Evaluating with occlusion mask v1...


100%|██████████| 11/11 [03:31<00:00, 19.25s/it]


Evaluating with occlusion mask v2...


100%|██████████| 11/11 [03:30<00:00, 19.17s/it]


...

Evaluating with occlusion mask v1...


100%|██████████| 10/10 [01:27<00:00,  8.77s/it]


Evaluating with occlusion mask v2...


100%|██████████| 10/10 [01:27<00:00,  8.73s/it]


,accuracy,precision,recall,f1_score,cohen_kappa,dataset,occlusion_mask,model,trial
0,0.775026,0.796729,0.775026,0.764436,0.672510,sleepedf,v1,ProtoSleepNetV1,0.0
1,0.685689,0.787584,0.685689,0.699871,0.569450,sleepedf,v1,SleepTransformer,NaN
2,0.729879,0.726222,0.729879,0.692671,0.597734,sleepedf,v2,ProtoSleepNetV1,0.0
3,0.476698,0.500948,0.476698,0.414760,0.243790,sleepedf,v2,SleepTransformer,NaN
4,0.784114,0.799203,0.784114,0.773170,0.685716,sleepedf,v1,ProtoSleepNetV1,1.0


In [9]:
results_df.to_csv("robustness_results.csv", index=False)